In [2]:
from __future__ import annotations

import copy
import json
from dataclasses import dataclass
from itertools import combinations

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import normalized_mutual_info_score

from pymoo.core.population import Population
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga3 import ReferenceDirectionSurvival
from pymoo.util.ref_dirs import get_reference_directions


EPS = 1e-12


# ============================================================
# Section 1: Configuration and Common Utilities
# ============================================================

@dataclass
class Config:
    """Configuration class for all algorithm parameters."""
    seed: int = 42

    # Hierarchy, role and coarsening parameters
    lambda_H: float = 0.5
    tau_H: float = 0.8
    delta_R: float = 0.10
    gamma: float = 0.5
    alpha: float = 0.3
    theta_recoal: float = 0.3

    # Feasibility thresholds
    xi_H: float = 0.05
    xi_R: float = 0.05

    # Representation learning parameters
    hidden_dim: int = 16
    embedding_dim: int = 8
    gcn_layers: int = 2
    sample_size: int = 64
    rounds: int = 2
    local_steps: int = 1
    batch_size: int = 4
    learning_rate: float = 0.02
    beta_time: float = 0.1
    beta_layer: float = 0.1
    clip_norm: float = 1.0
    noise_multiplier: float = 0.5

    # Evolutionary search parameters
    population_size: int = 24
    generations: int = 12
    Kmax: int = 32
    crossover_prob: float = 0.9
    mutation_prob: float = 0.15
    reference_partitions: int = 5

    # Objective and preference parameters
    lambda_balance: float = 0.2
    eta: float = 0.5


def canonical_labels(labels):
    """Normalize labels to consecutive integers based on first appearance order."""
    mapping = {}
    output = []
    for value in labels:
        value = int(value)
        if value not in mapping:
            mapping[value] = len(mapping)
        output.append(mapping[value])
    return np.asarray(output, dtype=int)


def canonical_matrix(S):
    """Normalize community labels for each layer separately."""
    return np.column_stack([
        canonical_labels(S[:, layer])
        for layer in range(S.shape[1])
    ])


def majority(values):
    """Majority voting; ties are resolved by selecting the smaller label."""
    labels, counts = np.unique(values, return_counts=True)
    return int(labels[np.argmax(counts)])


def safe_nmi(a, b):
    """Temporal objective on empty sets uses neutral convention 1."""
    if len(a) == 0:
        return 1.0
    return float(normalized_mutual_info_score(a, b))


def cosine(a, b):
    """Compute cosine similarity between two vectors."""
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denominator) if denominator > EPS else 0.0


def hierarchy_from_paths(paths, beta=0.5):
    """
    Construct hierarchy matrix from equal-depth leaf unit paths.
    
    Example path:
        ["Company", "R&D Department", "Platform Group"]
    
    Different employees in the same leaf unit share the same path.
    Root node has depth 0.
    """
    if not 0 < beta < 1:
        raise ValueError("beta must be in (0, 1).")
    if not paths or any(len(p) != len(paths[0]) for p in paths):
        raise ValueError("Paths must be non-empty and have the same depth.")
    if not paths[0] or any(p[0] != paths[0][0] for p in paths):
        raise ValueError("All paths must share the same root node.")

    n = len(paths)
    max_depth = len(paths[0]) - 1
    H = np.ones((n, n), dtype=float)

    for i in range(n):
        for j in range(i + 1, n):
            common = 0
            for a, b in zip(paths[i], paths[j]):
                if a != b:
                    break
                common += 1
            lca_depth = common - 1
            H[i, j] = H[j, i] = beta ** (max_depth - lca_depth)
    return H


# ============================================================
# Section 2: GCN and Federated Training Simulation with Clipping and Noise
# ============================================================

class GCN(nn.Module):
    """Graph Convolutional Network for embedding learning."""
    
    def __init__(self, input_dim, hidden_dim, output_dim, layers):
        super().__init__()
        if layers < 1:
            raise ValueError("GCN layers must be at least 1.")

        dimensions = (
            [input_dim]
            + [hidden_dim] * (layers - 1)
            + [output_dim]
        )
        self.layers = nn.ModuleList([
            nn.Linear(dimensions[i], dimensions[i + 1], bias=False)
            for i in range(layers)
        ])

    def forward(self, A, X):
        """Forward pass with symmetric normalization and self-loops."""
        # Add self-loops and perform symmetric normalization
        A_hat = A + torch.eye(A.shape[0], device=A.device, dtype=A.dtype)
        degrees = A_hat.sum(dim=1).clamp_min(1e-12)
        inverse = degrees.rsqrt()
        normalized = inverse[:, None] * A_hat * inverse[None, :]

        Z = X
        for index, layer in enumerate(self.layers):
            Z = layer(normalized @ Z)
            if index + 1 < len(self.layers):
                Z = F.relu(Z)
        return Z


class FederatedEmbedding:
    """
    Federated parameter averaging simulation in single machine.
    
    Client training only uses local induced subgraphs.
    Cross-client edges do not enter local GCN, but centralized structure evaluation retains these edges.
    
    Note:
    Per-anchor gradient clipping with noise does not constitute certified graph node-level differential privacy.
    """

    def __init__(self, cfg, rng):
        self.cfg = cfg
        self.rng = rng
        self.model = None
        self.input_dim = None

    def _ensure_model(self, input_dim):
        """Initialize model if not yet created, or verify input dimension consistency."""
        if self.model is None:
            self.input_dim = input_dim
            self.model = GCN(
                input_dim,
                self.cfg.hidden_dim,
                self.cfg.embedding_dim,
                self.cfg.gcn_layers,
            )
        elif input_dim != self.input_dim:
            raise ValueError("Different snapshots must use the same feature dimension.")

    def _anchor_loss(
        self, model, adjacency, features, anchor,
        global_indices, ids, previous_embeddings
    ):
        """One anchor corresponds to one clipping unit; not mislabeled as node-level DP unit."""
        Z = [model(A, X) for A, X in zip(adjacency, features)]
        zero = sum(z.sum() * 0.0 for z in Z)
        link_loss = zero
        time_loss = zero
        layer_loss = zero

        node_id = ids[global_indices[anchor]]

        for layer, (A, z) in enumerate(zip(adjacency, Z)):
            positive = torch.where(A[anchor] > 0)[0]
            positive = positive[positive != anchor]

            if len(positive):
                positive_logits = z[positive] @ z[anchor]
                link_loss = link_loss + (
                    A[anchor, positive] * F.softplus(-positive_logits)
                ).sum()

                # Sample negative nodes with replacement when insufficient non-neighbors
                negative_candidates = torch.where(A[anchor] <= 0)[0]
                negative_candidates = negative_candidates[
                    negative_candidates != anchor
                ]
                if len(negative_candidates):
                    sampled = self.rng.choice(
                        negative_candidates.cpu().numpy(),
                        size=5 * len(positive),
                        replace=True,
                    )
                    negative = torch.as_tensor(sampled, dtype=torch.long)
                    negative_logits = z[negative] @ z[anchor]
                    link_loss = link_loss + F.softplus(negative_logits).sum()

            if node_id in previous_embeddings:
                old = torch.as_tensor(
                    previous_embeddings[node_id][layer],
                    dtype=z.dtype,
                )
                time_loss = time_loss + ((z[anchor] - old) ** 2).sum()

        pairs = list(combinations(range(len(Z)), 2))
        for first, second in pairs:
            similarity = F.cosine_similarity(
                Z[first][anchor].unsqueeze(0),
                Z[second][anchor].unsqueeze(0),
                dim=1,
                eps=1e-8,
            )[0]
            layer_loss = layer_loss + 1.0 - similarity

        if pairs:
            layer_loss = layer_loss / len(pairs)

        return (
            link_loss / len(Z)
            + self.cfg.beta_time * time_loss / len(Z)
            + self.cfg.beta_layer * layer_loss
        )

    def fit_transform(self, snap, weighted_A, previous_embeddings):
        """Train federated model and produce embeddings for all nodes."""
        cfg = self.cfg
        n = len(snap["ids"])
        L = len(weighted_A)
        self._ensure_model(snap["X"][0].shape[1])

        client_groups = [
            np.where(snap["clients"] == client)[0]
            for client in np.unique(snap["clients"])
        ]

        for _ in range(cfg.rounds):
            states = []
            client_weights = []

            for nodes in client_groups:
                local_model = copy.deepcopy(self.model)
                parameters = list(local_model.parameters())

                for _ in range(cfg.local_steps):
                    # Same local induced subgraph used for all layer propagation and identity alignment
                    mean_A = np.mean([
                        A[np.ix_(nodes, nodes)] for A in weighted_A
                    ], axis=0)
                    degree = mean_A.sum(axis=1)
                    probabilities = (degree + 1e-6) ** 2
                    probabilities /= probabilities.sum()

                    size = min(cfg.sample_size, len(nodes))
                    sampled_local = self.rng.choice(
                        len(nodes), size=size, replace=False,
                        p=probabilities,
                    )
                    sampled_nodes = nodes[sampled_local]

                    adjacency = [
                        torch.as_tensor(
                            A[np.ix_(sampled_nodes, sampled_nodes)],
                            dtype=torch.float32,
                        )
                        for A in weighted_A
                    ]
                    features = [
                        torch.as_tensor(
                            X[sampled_nodes], dtype=torch.float32
                        )
                        for X in snap["X"]
                    ]

                    B = min(cfg.batch_size, size)
                    anchors = self.rng.choice(size, size=B, replace=False)
                    gradient_sums = [
                        torch.zeros_like(parameter)
                        for parameter in parameters
                    ]

                    # Compute gradients per anchor and clip full parameter gradient vector
                    for anchor in anchors:
                        loss = self._anchor_loss(
                            local_model, adjacency, features,
                            int(anchor), sampled_nodes, snap["ids"],
                            previous_embeddings,
                        )
                        gradients = torch.autograd.grad(loss, parameters)
                        norm = torch.sqrt(sum(
                            (gradient ** 2).sum()
                            for gradient in gradients
                        ))
                        factor = min(
                            1.0,
                            cfg.clip_norm / (float(norm) + EPS),
                        )
                        for total, gradient in zip(
                            gradient_sums, gradients
                        ):
                            total.add_(gradient.detach(), alpha=factor)

                    # Add Gaussian noise to gradient sum, then divide by actual batch size
                    with torch.no_grad():
                        for parameter, total in zip(
                            parameters, gradient_sums
                        ):
                            noise = torch.randn_like(total) * (
                                cfg.noise_multiplier * cfg.clip_norm
                            )
                            gradient = (total + noise) / B
                            parameter.add_(
                                gradient, alpha=-cfg.learning_rate
                            )

                states.append({
                    key: value.detach().clone()
                    for key, value in local_model.state_dict().items()
                })
                client_weights.append(len(nodes) / n)

            # Server aggregates model parameters only during this training step
            aggregated = {
                key: sum(
                    weight * state[key]
                    for weight, state in zip(client_weights, states)
                )
                for key in states[0]
            }
            self.model.load_state_dict(aggregated)

        # Client local inference; centralized simulation collects embeddings for subsequent algorithm use
        embeddings = [
            np.zeros((n, cfg.embedding_dim), dtype=float)
            for _ in range(L)
        ]
        self.model.eval()
        with torch.no_grad():
            for nodes in client_groups:
                for layer in range(L):
                    A = torch.as_tensor(
                        weighted_A[layer][np.ix_(nodes, nodes)],
                        dtype=torch.float32,
                    )
                    X = torch.as_tensor(
                        snap["X"][layer][nodes], dtype=torch.float32
                    )
                    embeddings[layer][nodes] = self.model(A, X).numpy()

        return embeddings


# ============================================================
# Section 3: DD-TriECD Main Algorithm
# ============================================================

class DDTriECD:
    """Dynamic Directed Triple Evolutionary Community Detection algorithm."""

    def __init__(self, cfg=None):
        self.cfg = cfg or Config()
        self._validate_config()

        np.random.seed(self.cfg.seed)
        torch.manual_seed(self.cfg.seed)
        self.rng = np.random.default_rng(self.cfg.seed)

        self.embedding = FederatedEmbedding(self.cfg, self.rng)
        self.weights = np.ones(3, dtype=float) / 3.0
        self.previous = None

        self.reference_directions = get_reference_directions(
            "das-dennis",
            3,
            n_partitions=self.cfg.reference_partitions,
        )
        self.problem = Problem(n_var=1, n_obj=3)

    def _validate_config(self):
        """Validate configuration parameters."""
        c = self.cfg
        if not 0 < c.gamma <= 1:
            raise ValueError("gamma must be in (0, 1].")
        if not 0 < c.tau_H <= 1:
            raise ValueError("tau_H must be in (0, 1].")
        if min(
            c.population_size, c.Kmax, c.sample_size,
            c.batch_size, c.rounds, c.local_steps,
            c.reference_partitions,
        ) < 1:
            raise ValueError("Population, sampling and training iteration parameters must be positive integers.")
        if c.clip_norm <= 0 or c.noise_multiplier < 0:
            raise ValueError("Clipping threshold must be positive, noise multiplier cannot be negative.")
        if min(c.delta_R, c.xi_H, c.xi_R) < 0:
            raise ValueError("Constraint tolerances cannot be negative.")
        if not all(0 <= v <= 1 for v in [
            c.crossover_prob, c.mutation_prob,
            c.lambda_balance, c.theta_recoal,
        ]):
            raise ValueError("Probability or ratio parameters must be in [0, 1].")

    def _prepare(self, snapshot):
        """Copy and validate current snapshot; array row order must match ids exactly."""
        snap = dict(snapshot)
        snap["ids"] = list(snapshot["ids"])
        n = len(snap["ids"])

        if n == 0 or len(set(snap["ids"])) != n:
            raise ValueError("ids must be non-empty and unique.")

        snap["A"] = [np.asarray(A, dtype=float) for A in snapshot["A"]]
        snap["X"] = [np.asarray(X, dtype=float) for X in snapshot["X"]]
        snap["H"] = np.asarray(snapshot["H"], dtype=float)

        for key in ["roles", "units", "clients"]:
            snap[key] = np.asarray(snapshot[key], dtype=int)
            if snap[key].shape != (n,):
                raise ValueError(f"{key} length must equal node count.")

        snap["templates"] = {
            int(key): np.asarray(value, dtype=float)
            for key, value in snapshot["templates"].items()
        }

        if not snap["A"] or len(snap["A"]) != len(snap["X"]):
            raise ValueError("A and X must contain the same number of non-empty network layers.")

        dimension = snap["X"][0].shape[1]
        for A, X in zip(snap["A"], snap["X"]):
            if A.shape != (n, n):
                raise ValueError("Adjacency matrix dimension error.")
            if not np.isfinite(A).all() or np.any(A < 0):
                raise ValueError("Adjacency matrix must be finite and non-negative.")
            if not np.allclose(A, A.T):
                raise ValueError("This implementation targets undirected graphs; adjacency matrix must be symmetric.")
            if not np.allclose(np.diag(A), 0):
                raise ValueError("Input graph must remove self-loops; GCN adds self-loops automatically.")
            if X.shape != (n, dimension) or not np.isfinite(X).all():
                raise ValueError("Feature dimensions must be consistent across layers and numerically finite.")

        H = snap["H"]
        if (
            H.shape != (n, n)
            or not np.isfinite(H).all()
            or np.any(H <= 0)
            or np.any(H > 1)
            or not np.allclose(H, H.T)
        ):
            raise ValueError("H must be a symmetric matrix with values in (0, 1].")

        if not snap["templates"]:
            raise ValueError("Role templates must be provided.")
        role_count = len(next(iter(snap["templates"].values())))
        if np.any(snap["roles"] < 0) or np.any(snap["roles"] >= role_count):
            raise ValueError("Roles must be encoded as consecutive integers within template range.")

        for unit in np.unique(snap["units"]):
            if int(unit) not in snap["templates"]:
                raise ValueError(f"Unit {unit} missing role template.")

        for template in snap["templates"].values():
            if (
                template.shape != (role_count,)
                or not np.isfinite(template).all()
                or np.any(template < 0)
                or not np.isclose(template.sum(), 1)
            ):
                raise ValueError("Role templates must be probability vectors of same dimension.")

        if self.previous is not None:
            if len(snap["A"]) != self.previous["labels"].shape[1]:
                raise ValueError("This implementation requires consistent layer count and semantics across snapshots.")

        self.snap = snap
        self.n = n
        self.L = len(snap["A"])
        self.role_count = role_count
        self.A = [
            A * (1.0 + self.cfg.lambda_H * H)
            for A in snap["A"]
        ]
        self.mean_A = np.mean(self.A, axis=0)

    # --------------------------------------------------------
    # Constraint Evaluation
    # --------------------------------------------------------

    def _role_distance(self, members):
        """Return L1 distance between community role profile and majority business unit template."""
        if len(members) == 0:
            return 0.0
        members = np.asarray(members, dtype=int)
        unit = majority(self.snap["units"][members])
        counts = np.bincount(
            self.snap["roles"][members],
            minlength=self.role_count,
        )
        profile = counts / len(members)
        return float(np.abs(profile - self.snap["templates"][unit]).sum())

    def _resource_excess(self, members):
        """Compute resource constraint violation."""
        if len(members) == 0:
            return 0.0
        return max(0.0, self._role_distance(members) - self.cfg.delta_R)

    def _hierarchy_ok(self, members):
        """Check if all members satisfy hierarchy constraint."""
        if len(members) <= 1:
            return True
        H = self.snap["H"][np.ix_(members, members)]
        return bool(np.all(H >= self.cfg.tau_H - EPS))

    def _matchable(self, members):
        """Check if members can be merged into a super-node."""
        return (
            self._hierarchy_ok(members)
            and self._role_distance(members) <= self.cfg.delta_R + EPS
        )

    def _layer_constraints(self, labels):
        """Exact computation on original nodes to avoid ignoring super-node sizes."""
        bad_pairs = 0
        total_pairs = 0
        resource = []

        for label in np.unique(labels):
            members = np.where(labels == label)[0]
            size = len(members)
            total_pairs += size * (size - 1) // 2

            if size > 1:
                H = self.snap["H"][np.ix_(members, members)]
                bad_pairs += int(
                    np.triu(H < self.cfg.tau_H - EPS, k=1).sum()
                )
            resource.append(self._resource_excess(members))

        cvh = bad_pairs / total_pairs if total_pairs else 0.0
        cvr = float(np.mean(resource)) if resource else 0.0
        return float(cvh), cvr, bad_pairs

    # --------------------------------------------------------
    # Coarsening and Warm Start
    # --------------------------------------------------------

    def _group_score(self, first, second):
        """Compute merge score between two groups."""
        edge = self.mean_A[np.ix_(first, second)].sum()
        first_degree = self.mean_A[first].sum()
        second_degree = self.mean_A[second].sum()
        structural = edge / np.sqrt(
            max(first_degree * second_degree, EPS)
        )
        first_embedding = self.concat_Z[first].mean(axis=0)
        second_embedding = self.concat_Z[second].mean(axis=0)
        return structural + self.cfg.alpha * cosine(
            first_embedding, second_embedding
        )

    def _greedy_coarsen(self, groups):
        """Greedy coarsening to target size."""
        target = max(1, int(np.floor(self.cfg.gamma * self.n)))
        groups = [list(group) for group in groups]

        while len(groups) > target:
            order = sorted(
                range(len(groups)),
                key=lambda index: (
                    -self.mean_A[groups[index]].sum(),
                    min(groups[index]),
                ),
            )
            used = set()
            merged = []
            reductions = 0

            for u in order:
                if u in used:
                    continue

                candidates = []
                if len(groups) - reductions > target:
                    for v in order:
                        if v == u or v in used:
                            continue
                        if self.mean_A[np.ix_(groups[u], groups[v])].sum() <= 0:
                            continue
                        if self._matchable(groups[u] + groups[v]):
                            candidates.append(v)

                if candidates:
                    v = max(
                        candidates,
                        key=lambda value: self._group_score(
                            groups[u], groups[value]
                        ),
                    )
                    merged.append(sorted(groups[u] + groups[v]))
                    used.update([u, v])
                    reductions += 1
                else:
                    merged.append(groups[u])
                    used.add(u)

            groups = merged
            if reductions == 0:
                # Stop when constraints prevent further merging; do not force reaching gamma
                break

        return sorted(groups, key=min)

    def _coarsen(self):
        """Perform graph coarsening with warm start from previous snapshot."""
        rebuilt = False
        change_ratio = 0.0

        if self.previous is None:
            groups = [[i] for i in range(self.n)]
        else:
            current_index = {
                node_id: index
                for index, node_id in enumerate(self.snap["ids"])
            }
            groups = []
            assigned = set()

            for old_group in self.previous["groups"]:
                members = [
                    current_index[node_id]
                    for node_id in old_group
                    if node_id in current_index
                ]
                if not members:
                    continue

                # Departures or role changes may invalidate old super-node merge conditions
                if len(members) == 1 or self._matchable(members):
                    groups.append(members)
                else:
                    groups.extend([[member] for member in members])
                assigned.update(members)

            # New nodes enter best matchable adjacent super-node, otherwise create separate super-node
            for node in range(self.n):
                if node in assigned:
                    continue
                candidates = [
                    index
                    for index, group in enumerate(groups)
                    if self.mean_A[node, group].sum() > 0
                    and self._matchable(group + [node])
                ]
                if candidates:
                    selected = max(
                        candidates,
                        key=lambda index: self._group_score(
                            [node], groups[index]
                        ),
                    )
                    groups[selected].append(node)
                else:
                    groups.append([node])

            # Estimate whether optimal super-node for persistent nodes has changed
            owner = {
                node: index
                for index, group in enumerate(groups)
                for node in group
            }
            old_ids = set(self.previous["ids"])
            changed = 0
            persistent_count = 0

            for node, node_id in enumerate(self.snap["ids"]):
                if node_id not in old_ids:
                    continue
                persistent_count += 1
                own = owner[node]
                own_rest = [v for v in groups[own] if v != node]
                own_score = (
                    self._group_score([node], own_rest)
                    if own_rest else -np.inf
                )

                best_group = own
                best_score = own_score
                for index, group in enumerate(groups):
                    if index == own:
                        continue
                    if (
                        self.mean_A[node, group].sum() > 0
                        and self._matchable(group + [node])
                    ):
                        score = self._group_score([node], group)
                        if score > best_score + EPS:
                            best_group = index
                            best_score = score

                changed += int(best_group != own)

            change_ratio = changed / max(persistent_count, 1)
            if change_ratio > self.cfg.theta_recoal:
                groups = [[i] for i in range(self.n)]
                rebuilt = True

        self.groups = self._greedy_coarsen(groups)
        self.ns = len(self.groups)
        self.assignment = np.empty(self.n, dtype=int)

        for index, members in enumerate(self.groups):
            self.assignment[members] = index

        self.P = np.eye(self.ns, dtype=float)[self.assignment]
        self.Ac = [self.P.T @ A @ self.P for A in self.A]
        sizes = self.P.sum(axis=0)
        self.Xc = [
            (self.P.T @ X) / sizes[:, None]
            for X in self.snap["X"]
        ]

        self.Hc = np.ones((self.ns, self.ns))
        self.Rc = np.zeros((self.ns, self.role_count))
        for u, members in enumerate(self.groups):
            self.Rc[u] = np.bincount(
                self.snap["roles"][members],
                minlength=self.role_count,
            ) / len(members)
            for v in range(u + 1, self.ns):
                value = self.snap["H"][
                    np.ix_(members, self.groups[v])
                ].min()
                self.Hc[u, v] = self.Hc[v, u] = value

        self.coarsening_info = {
            "original_nodes": self.n,
            "super_nodes": self.ns,
            "target_reached": (
                self.ns <= max(1, int(np.floor(self.cfg.gamma * self.n)))
            ),
            "rebuilt": rebuilt,
            "change_ratio": float(change_ratio),
        }

    # --------------------------------------------------------
    # Three Objective Functions
    # --------------------------------------------------------

    @staticmethod
    def _modularity(A, labels):
        """Compute modularity for a single layer."""
        volume = float(A.sum())
        if volume <= EPS:
            return 0.0
        degree = A.sum(axis=1)
        Q = 0.0
        for label in np.unique(labels):
            members = np.where(labels == label)[0]
            internal = A[np.ix_(members, members)].sum()
            total_degree = degree[members].sum()
            Q += internal / volume - (total_degree / volume) ** 2
        return float(Q)

    @staticmethod
    def _size_distribution(labels):
        """Compute sorted community size distribution."""
        counts = np.unique(labels, return_counts=True)[1]
        return np.sort(counts.astype(float))[::-1] / len(labels)

    def _objectives(self, S):
        """Compute three objectives for a candidate solution."""
        labels = S[self.assignment]
        q_layers = [
            self._modularity(self.Ac[layer], S[:, layer])
            for layer in range(self.L)
        ]

        agreement = []
        for first, second in combinations(range(self.L), 2):
            nmi = safe_nmi(labels[:, first], labels[:, second])
            p = self._size_distribution(labels[:, first])
            q = self._size_distribution(labels[:, second])
            size = max(len(p), len(q))
            p = np.pad(p, (0, size - len(p)))
            q = np.pad(q, (0, size - len(q)))

            # scipy returns JSD distance; square to get divergence
            jsd = float(jensenshannon(p, q, base=2.0) ** 2)
            agreement.append(
                (1 - self.cfg.lambda_balance) * nmi
                + self.cfg.lambda_balance * (1 - jsd)
            )

        inter = float(np.mean(agreement)) if agreement else 1.0

        if self.previous is None:
            temporal = 1.0
        else:
            old_index = {
                node_id: index
                for index, node_id in enumerate(self.previous["ids"])
            }
            current_positions = []
            previous_positions = []
            for index, node_id in enumerate(self.snap["ids"]):
                if node_id in old_index:
                    current_positions.append(index)
                    previous_positions.append(old_index[node_id])

            temporal = float(np.mean([
                safe_nmi(
                    labels[current_positions, layer],
                    self.previous["labels"][previous_positions, layer],
                )
                for layer in range(self.L)
            ]))

        return np.array([
            np.mean(q_layers), inter, temporal
        ], dtype=float), q_layers

    def _evaluate(self, S):
        """Evaluate a candidate solution."""
        S = canonical_matrix(S)
        labels = S[self.assignment]
        objectives, q_layers = self._objectives(S)
        constraints = [
            self._layer_constraints(labels[:, layer])[:2]
            for layer in range(self.L)
        ]

        # Use layer-wise maximum: all layers must satisfy thresholds for feasibility
        cvh = max(value[0] for value in constraints)
        cvr = max(value[1] for value in constraints)
        feasible = (
            cvh <= self.cfg.xi_H + EPS
            and cvr <= self.cfg.xi_R + EPS
        )

        return {
            "S": S,
            "f": objectives,
            "q_layers": q_layers,
            "cvh": cvh,
            "cvr": cvr,
            "cv": cvh + cvr,
            "feasible": feasible,
            "layer_constraints": constraints,
        }

    # --------------------------------------------------------
    # Initialization, Crossover, Mutation and Repair
    # --------------------------------------------------------

    def _label_propagation(self):
        """Initialize population using label propagation."""
        S = np.zeros((self.ns, self.L), dtype=int)

        for layer in range(self.L):
            labels = np.arange(self.ns) % self.cfg.Kmax

            # Partial initialization retains individual super-nodes for population diversity
            if self.rng.random() >= 0.25:
                for _ in range(6):
                    changed = False
                    for u in self.rng.permutation(self.ns):
                        weights = self.Ac[layer][u].copy()
                        weights[u] = 0
                        neighbors = np.where(weights > 0)[0]
                        if not len(neighbors):
                            continue

                        totals = {}
                        for v in neighbors:
                            label = int(labels[v])
                            totals[label] = totals.get(label, 0.0) + weights[v]

                        maximum = max(totals.values())
                        options = [
                            label for label, weight in totals.items()
                            if abs(weight - maximum) <= EPS
                        ]
                        new_label = int(self.rng.choice(options))
                        changed |= new_label != labels[u]
                        labels[u] = new_label
                    if not changed:
                        break

            S[:, layer] = canonical_labels(labels)
        return S

    def _project_previous(self, old_labels, fallback):
        """Project old population through identity intersection of persistent nodes."""
        index = {
            node_id: position
            for position, node_id in enumerate(self.previous["ids"])
        }
        S = fallback.copy()

        for u, members in enumerate(self.groups):
            positions = [
                index[self.snap["ids"][member]]
                for member in members
                if self.snap["ids"][member] in index
            ]
            if positions:
                for layer in range(self.L):
                    S[u, layer] = majority(old_labels[positions, layer])
        return canonical_matrix(S)

    def _align_parent(self, reference, moving):
        """Align second parent through Hungarian matching; per-layer label mapping stays injective."""
        output = moving.copy()

        for layer in range(self.L):
            first = np.unique(reference[:, layer])
            second = np.unique(moving[:, layer])
            confusion = np.zeros((len(first), len(second)), dtype=int)

            for i, a in enumerate(first):
                for j, b in enumerate(second):
                    confusion[i, j] = np.sum(
                        (reference[:, layer] == a)
                        & (moving[:, layer] == b)
                    )

            rows, columns = linear_sum_assignment(-confusion)
            mapping = {
                int(second[j]): int(first[i])
                for i, j in zip(rows, columns)
            }
            unused = [
                label for label in range(self.cfg.Kmax)
                if label not in mapping.values()
            ]

            for label in second:
                if int(label) not in mapping:
                    mapping[int(label)] = unused.pop(0)

            output[:, layer] = [
                mapping[int(label)] for label in moving[:, layer]
            ]
        return output

    def _mutate(self, S):
        """Perform mutation operation."""
        result = S.copy()

        for layer in range(self.L):
            for u in self.rng.permutation(self.ns):
                if self.rng.random() >= self.cfg.mutation_prob:
                    continue

                neighbors = np.where(self.Ac[layer][u] > 0)[0]
                neighbors = neighbors[neighbors != u]
                if not len(neighbors):
                    continue

                v = int(self.rng.choice(neighbors))
                target = result[v, layer]
                if target == result[u, layer]:
                    continue

                labels = result[:, layer][self.assignment]
                before = np.where(labels == target)[0]
                after = np.concatenate([before, self.groups[u]])

                # Per original description, only restrict receiving community resource violation not increasing
                if (
                    self._resource_excess(after)
                    <= self._resource_excess(before) + EPS
                ):
                    result[u, layer] = target

        return canonical_matrix(result)

    def _repair(self, S):
        """
        Perform one pass of super-node traversal per layer.
        
        Accepting a move requires:
        1. Receiving community hierarchy feasible;
        2. Hierarchy violation pairs strictly decrease;
        3. Current layer overall CVR does not increase.
        
        Keep violations when no legal destination found; final output explicitly reports them.
        """
        result = S.copy()

        for layer in range(self.L):
            for u in range(self.ns):
                labels = result[:, layer][self.assignment]
                old_label = result[u, layer]
                members_u = self.groups[u]
                other = np.where(labels == old_label)[0]
                other = [
                    int(node) for node in other
                    if node not in members_u
                ]

                if not other:
                    continue
                H = self.snap["H"][np.ix_(members_u, other)]
                if np.all(H >= self.cfg.tau_H - EPS):
                    continue

                _, old_cvr, old_bad = self._layer_constraints(labels)
                existing = list(np.unique(result[:, layer]))
                fresh = next(
                    (
                        label for label in range(self.cfg.Kmax)
                        if label not in existing
                    ),
                    None,
                )
                candidates = existing + (
                    [fresh] if fresh is not None else []
                )
                valid_moves = []

                for target in candidates:
                    if target == old_label:
                        continue
                    receiving = np.where(labels == target)[0].tolist()
                    if not self._hierarchy_ok(receiving + members_u):
                        continue

                    trial = result[:, layer].copy()
                    trial[u] = target
                    trial_labels = trial[self.assignment]
                    _, new_cvr, new_bad = self._layer_constraints(
                        trial_labels
                    )

                    if new_bad < old_bad and new_cvr <= old_cvr + EPS:
                        receiving_supernodes = np.where(
                            result[:, layer] == target
                        )[0]
                        internal_degree = self.Ac[layer][
                            u, receiving_supernodes
                        ].sum()
                        valid_moves.append((internal_degree, int(target)))

                if valid_moves:
                    _, target = max(valid_moves, key=lambda value: value[0])
                    result[u, layer] = target

        return canonical_matrix(result)

    def _initial_population(self):
        """Generate initial population."""
        population = []
        old_count = (
            self.cfg.population_size // 2
            if self.previous is not None else 0
        )

        for index in range(self.cfg.population_size):
            S = self._label_propagation()
            if index < old_count:
                old = self.previous["population"][
                    index % len(self.previous["population"])
                ]
                S = self._project_previous(old, S)
            population.append(self._evaluate(self._repair(S)))
        return population

    # --------------------------------------------------------
    # Constraint Dominance and NSGA-III Environmental Selection
    # --------------------------------------------------------

    @staticmethod
    def _dominates(first, second):
        """Check if first dominates second."""
        if first["feasible"] != second["feasible"]:
            return first["feasible"]

        if not first["feasible"]:
            return first["cv"] < second["cv"] - EPS

        return (
            np.all(first["f"] >= second["f"] - EPS)
            and np.any(first["f"] > second["f"] + EPS)
        )

    def _tournament(self, population):
        """Perform tournament selection."""
        i, j = self.rng.integers(0, len(population), size=2)
        first, second = population[i], population[j]
        if self._dominates(first, second):
            return first
        if self._dominates(second, first):
            return second
        return first if self.rng.random() < 0.5 else second

    def _select(self, individuals, count):
        """Select survivors using NSGA-III or constraint ranking."""
        feasible = [
            index for index, value in enumerate(individuals)
            if value["feasible"]
        ]
        infeasible = [
            index for index, value in enumerate(individuals)
            if not value["feasible"]
        ]

        if len(feasible) > count:
            # pymoo uses minimization objectives, so negate the three maximization objectives
            objectives = -np.vstack([
                individuals[index]["f"] for index in feasible
            ])
            population = Population.new(
                F=objectives,
                original_index=np.asarray(feasible, dtype=int),
            )
            survivors = self.survival.do(
                self.problem,
                population,
                n_survive=count,
            )
            selected = [
                int(index)
                for index in survivors.get("original_index")
            ]
        else:
            selected = feasible.copy()

        if len(selected) < count:
            # Per original paper, use raw CVH + CVR, not threshold exceedance
            infeasible.sort(key=lambda index: individuals[index]["cv"])
            selected.extend(infeasible[:count - len(selected)])

        return [individuals[index] for index in selected]

    def _search(self):
        """Run evolutionary search."""
        # Reinitialize objective normalization state per snapshot
        self.survival = ReferenceDirectionSurvival(
            self.reference_directions
        )
        population = self._initial_population()

        for _ in range(self.cfg.generations):
            offspring = []

            while len(offspring) < self.cfg.population_size:
                first = self._tournament(population)["S"]
                second = self._tournament(population)["S"]
                child = first.copy()

                if self.rng.random() < self.cfg.crossover_prob:
                    second = self._align_parent(first, second)
                    mask = self.rng.random(first.shape) < 0.5
                    child = np.where(mask, first, second)

                child = self._mutate(child)
                child = self._repair(child)
                offspring.append(self._evaluate(child))

            population = self._select(
                population + offspring,
                self.cfg.population_size,
            )

        return population

    def _feasible_front(self, population):
        """Extract non-dominated feasible front."""
        feasible = [value for value in population if value["feasible"]]
        front = [
            value for value in feasible
            if not any(
                self._dominates(other, value)
                for other in feasible
            )
        ]

        # Remove completely duplicate partitions; keep structurally different ones with same objectives
        unique = {}
        for value in front:
            unique.setdefault(value["S"].tobytes(), value)
        return list(unique.values())

    # --------------------------------------------------------
    # Cross-layer Correspondence, Consensus and Final Decision
    # --------------------------------------------------------

    def _correspondence(self, labels, q_layers):
        """Establish cross-layer correspondence and compute consensus."""
        # Select layer with highest modularity as anchor for consensus IDs
        anchor = int(np.argmax(q_layers))
        reference = labels[:, anchor]
        reference_values = np.unique(reference)

        aligned = np.empty_like(labels)
        aligned[:, anchor] = reference
        maps = [None] * self.L
        maps[anchor] = {
            int(value): int(value) for value in reference_values
        }
        next_id = int(reference_values.max()) + 1

        for layer in range(self.L):
            if layer == anchor:
                continue
            values = np.unique(labels[:, layer])
            confusion = np.zeros(
                (len(reference_values), len(values)), dtype=int
            )
            for i, first in enumerate(reference_values):
                for j, second in enumerate(values):
                    confusion[i, j] = np.sum(
                        (reference == first)
                        & (labels[:, layer] == second)
                    )

            rows, columns = linear_sum_assignment(-confusion)
            mapping = {}

            for i, j in zip(rows, columns):
                if confusion[i, j] > 0:
                    mapping[int(values[j])] = int(reference_values[i])

            # Assign new IDs to unmatched communities without identity overlap to avoid incorrect merging
            for value in values:
                value = int(value)
                if value not in mapping:
                    mapping[value] = next_id
                    next_id += 1

            maps[layer] = mapping
            aligned[:, layer] = [
                mapping[int(value)] for value in labels[:, layer]
            ]

        consensus = []
        layer_order = np.argsort(-np.asarray(q_layers), kind="stable")

        for row in aligned:
            values, counts = np.unique(row, return_counts=True)
            tied = set(values[counts == counts.max()].tolist())

            # Among tied candidates, prefer label from layer with highest modularity
            winner = next(
                int(row[layer])
                for layer in layer_order
                if int(row[layer]) in tied
            )
            consensus.append(winner)

        return (
            aligned,
            canonical_labels(consensus),
            maps,
            anchor,
        )

    def fit_snapshot(self, snapshot):
        """Process a single snapshot."""
        self._prepare(snapshot)

        previous_embeddings = (
            self.previous["embeddings"]
            if self.previous is not None else {}
        )
        self.Z = self.embedding.fit_transform(
            self.snap, self.A, previous_embeddings
        )
        self.concat_Z = np.concatenate(self.Z, axis=1)

        self._coarsen()
        population = self._search()
        front = self._feasible_front(population)
        used_weights = self.weights.copy()

        if front:
            objectives = np.vstack([value["f"] for value in front])
            ideal = objectives.max(axis=0)
            nadir = objectives.min(axis=0)
            deficits = (ideal - objectives) / (ideal - nadir + EPS)
            asf = np.max(deficits * self.weights[None, :], axis=1)
            chosen_index = int(np.argmin(asf))
            chosen = front[chosen_index]
            deficit = deficits[chosen_index]

            # Use numerically stable multiplicative weight update
            log_weights = (
                np.log(self.weights + EPS) + self.cfg.eta * deficit
            )
            log_weights -= log_weights.max()
            self.weights = np.exp(log_weights)
            self.weights /= self.weights.sum()
            status = "feasible_found"
        else:
            chosen = min(population, key=lambda value: value["cv"])
            deficit = None
            status = "no_feasible_solution_found"
            # Do not update preferences using infeasible objectives when no feasible front exists

        labels = chosen["S"][self.assignment]
        aligned, consensus, maps, anchor = self._correspondence(
            labels, chosen["q_layers"]
        )

        # Majority voting does not guarantee constraint preservation; audit consensus partition separately
        consensus_h, consensus_r, _ = self._layer_constraints(consensus)
        consensus_feasible = (
            consensus_h <= self.cfg.xi_H + EPS
            and consensus_r <= self.cfg.xi_R + EPS
        )

        warnings = []
        if not front:
            warnings.append(
                "Search found no feasible solution; this does not prove the constrained problem has no solution."
            )
        if not self.coarsening_info["target_reached"]:
            warnings.append("Constraints prevented further coarsening; target retention ratio not reached.")
        if not consensus_feasible:
            warnings.append(
                "Consensus partition failed constraint audit; cannot be marked as deployable partition."
            )

        output = {
            "ids": self.snap["ids"],
            "status": status,
            "layer_labels": labels.tolist(),
            "aligned_layer_labels": aligned.tolist(),
            "consensus_labels": consensus.tolist(),
            "cross_layer_maps": maps,
            "anchor_layer": anchor,
            "objectives": {
                "Q_intra": float(chosen["f"][0]),
                "S_inter": float(chosen["f"][1]),
                "S_temp": float(chosen["f"][2]),
            },
            "violations": {
                "CVH": float(chosen["cvh"]),
                "CVR": float(chosen["cvr"]),
            },
            "per_layer_violations": [
                {"CVH": float(h), "CVR": float(r)}
                for h, r in chosen["layer_constraints"]
            ],
            "consensus_audit": {
                "CVH": float(consensus_h),
                "CVR": float(consensus_r),
                "feasible": bool(consensus_feasible),
            },
            "weights_used": used_weights.tolist(),
            "weights_next": self.weights.tolist(),
            "deficit": None if deficit is None else deficit.tolist(),
            "pareto_size": len(front),
            "coarsening": self.coarsening_info,
            "privacy": {
                "mode": "centralized_research_simulation",
                "gradient_clipping_and_noise": True,
                "node_level_dp_certified": False,
                "epsilon": None,
            },
            "warnings": warnings,
        }

        # Preserve information indexed by global identity for next snapshot's entry and departure
        self.previous = {
            "ids": list(self.snap["ids"]),
            "groups": [
                [self.snap["ids"][node] for node in members]
                for members in self.groups
            ],
            "labels": labels.copy(),
            "population": [
                value["S"][self.assignment].copy()
                for value in population
            ],
            "embeddings": {
                node_id: np.stack([
                    self.Z[layer][index]
                    for layer in range(self.L)
                ])
                for index, node_id in enumerate(self.snap["ids"])
            },
        }

        return output

    def fit(self, snapshots):
        """Process multiple snapshots in chronological order."""
        return [self.fit_snapshot(snapshot) for snapshot in snapshots]


# ============================================================
# Section 4: Simple Input Data and Running Example
# ============================================================

def make_demo_snapshot(scale=1.0):
    """
    Construct 8 employees, 2 business units, 2 network layers.
    
    Business unit 0: employees 0, 1, 2, 3
    Business unit 1: employees 4, 5, 6, 7
    
    Role encoding:
        0: Manager
        1: Reviewer
    
    Each unit's target role ratio is [0.5, 0.5].
    """
    ids = list(range(8))

    # Two 4-node complete graphs, no edges across units
    A = np.zeros((8, 8), dtype=float)
    for members in [range(4), range(4, 8)]:
        for i in members:
            for j in members:
                if i != j:
                    A[i, j] = scale

    roles = np.array([0, 1, 0, 1, 0, 1, 0, 1])
    units = np.array([0, 0, 0, 0, 1, 1, 1, 1])

    # Two clients each hold one business unit
    clients = units.copy()

    paths = [
        ["Company", "Business Unit 0"] if unit == 0
        else ["Company", "Business Unit 1"]
        for unit in units
    ]
    H = hierarchy_from_paths(paths, beta=0.5)

    # Simple 2D features; both layers use same feature dimension
    X = np.array([
        [1.0, 0.0],
        [0.9, 0.1],
        [1.1, 0.0],
        [0.8, 0.2],
        [0.0, 1.0],
        [0.1, 0.9],
        [0.0, 1.1],
        [0.2, 0.8],
    ])

    return {
        "ids": ids,
        "A": [A, 0.7 * A],
        "X": [X, X.copy()],
        "H": H,
        "roles": roles,
        "units": units,
        "clients": clients,
        "templates": {
            0: np.array([0.5, 0.5]),
            1: np.array([0.5, 0.5]),
        },
    }


if __name__ == "__main__":
    # Small example to reduce thread scheduling overhead
    torch.set_num_threads(1)

    config = Config(
        seed=42,
        gamma=0.25,
        tau_H=0.8,
        delta_R=0.10,
        population_size=24,
        generations=8,
        rounds=2,
        noise_multiplier=0.5,
    )

    # Second snapshot only increases interaction strength to verify temporal continuity
    snapshots = [
        make_demo_snapshot(scale=1.0),
        make_demo_snapshot(scale=1.2),
    ]

    algorithm = DDTriECD(config)
    results = algorithm.fit(snapshots)

    for t, result in enumerate(results, start=1):
        summary = {
            "snapshot": t,
            "status": result["status"],
            "super_nodes": result["coarsening"]["super_nodes"],
            "layer_labels": result["layer_labels"],
            "consensus_labels": result["consensus_labels"],
            "objectives": {
                key: round(value, 4)
                for key, value in result["objectives"].items()
            },
            "violations": result["violations"],
            "consensus_audit": result["consensus_audit"],
            "weights_next": [
                round(value, 4) for value in result["weights_next"]
            ],
        }
        print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "snapshot": 1,
  "status": "feasible_found",
  "super_nodes": 2,
  "layer_labels": [
    [
      0,
      0
    ],
    [
      0,
      0
    ],
    [
      0,
      0
    ],
    [
      0,
      0
    ],
    [
      1,
      1
    ],
    [
      1,
      1
    ],
    [
      1,
      1
    ],
    [
      1,
      1
    ]
  ],
  "consensus_labels": [
    0,
    0,
    0,
    0,
    1,
    1,
    1,
    1
  ],
  "objectives": {
    "Q_intra": 0.5,
    "S_inter": 1.0,
    "S_temp": 1.0
  },
  "violations": {
    "CVH": 0.0,
    "CVR": 0.0
  },
  "consensus_audit": {
    "CVH": 0.0,
    "CVR": 0.0,
    "feasible": true
  },
  "weights_next": [
    0.3333,
    0.3333,
    0.3333
  ]
}
{
  "snapshot": 2,
  "status": "feasible_found",
  "super_nodes": 2,
  "layer_labels": [
    [
      0,
      0
    ],
    [
      0,
      0
    ],
    [
      0,
      0
    ],
    [
      0,
      0
    ],
    [
      1,
      1
    ],
    [
      1,
      1
    ],
    [
      1,
      1
    ],
    [
 